# Upload Dataset to Vayu Object Storage

In [ ]:
import os
# import boto3

from pathlib import Path

from dotenv import load_dotenv

_repo_root = Path.cwd()
if not (_repo_root / "requirements.txt").is_file():
    _repo_root = _repo_root.parent
load_dotenv(_repo_root / ".env")

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"


def _require_env(name: str) -> str:
    """Return a required env var or abort if missing or still a placeholder."""
    value = os.environ.get(name, "").strip()
    if not value:
        raise EnvironmentError(
            f"Missing required environment variable {name!r}. "
            "Copy .env.example to .env at the ask-it repo root and set a real value "
            "(see 01_dataset/README.md)."
        )
    if "<" in value or "***" in value:
        raise EnvironmentError(
            f"Environment variable {name!r} looks like a placeholder. "
            "Update ask-it/.env with real credentials."
        )
    return value


# Credentials and bucket from environment — never paste secrets into notebook cells
AWS_KEY = _require_env("VAYU_S3_KEY")
AWS_SECRET = _require_env("VAYU_S3_SECRET")
S3_ENDPOINT = _require_env("VAYU_S3_ENDPOINT")
BUCKET = _require_env("VAYU_S3_BUCKET")

# Local folder path to upload
LOCAL_FOLDER = "docs"  # Change this to your dataset folder

# S3 folder/destination path (target root folder in bucket)
S3_PREFIX = "ask-it/docs/"  # S3 'folder' (adds trailing slash)

# Create S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    endpoint_url=S3_ENDPOINT
)

def upload_folder(local_folder, bucket, s3_prefix):
    for root, _, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, local_folder)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")
            print(f"Uploading {local_path} → s3://{bucket}/{s3_key}")
            s3.upload_file(local_path, bucket, s3_key)
    print("DONE: Folder uploaded successfully")

upload_folder(LOCAL_FOLDER, BUCKET, S3_PREFIX)

# Download Dataset from Vayu Object Storage

In [ ]:
import os
import boto3

from pathlib import Path

from dotenv import load_dotenv

_repo_root = Path.cwd()
if not (_repo_root / "requirements.txt").is_file():
    _repo_root = _repo_root.parent
load_dotenv(_repo_root / ".env")

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"


def _require_env(name: str) -> str:
    """Return a required env var or abort if missing or still a placeholder."""
    value = os.environ.get(name, "").strip()
    if not value:
        raise EnvironmentError(
            f"Missing required environment variable {name!r}. "
            "Copy .env.example to .env at the ask-it repo root and set a real value "
            "(see 01_dataset/README.md)."
        )
    if "<" in value or "***" in value:
        raise EnvironmentError(
            f"Environment variable {name!r} looks like a placeholder. "
            "Update ask-it/.env with real credentials."
        )
    return value


# Credentials and bucket from environment — never paste secrets into notebook cells
AWS_KEY = _require_env("VAYU_S3_KEY")
AWS_SECRET = _require_env("VAYU_S3_SECRET")
S3_ENDPOINT = _require_env("VAYU_S3_ENDPOINT")
BUCKET = _require_env("VAYU_S3_BUCKET")

# Local folder path to download to
LOCAL_FOLDER = "docs"  # Change this to your target download folder

# S3 folder/source path (source root folder in bucket)
S3_PREFIX = "ask-it/docs/"  # S3 'folder' (adds trailing slash)

# Create S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    endpoint_url=S3_ENDPOINT
)

def download_folder(bucket, s3_prefix, local_folder):
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=s3_prefix):
        if "Contents" not in page:
            print("No objects found in S3 folder!")
            return
        for obj in page["Contents"]:
            key = obj["Key"]
            # Skip "directory" objects
            if key.endswith('/'):
                continue
            # Remove S3_PREFIX from key to get relative path
            rel_path = os.path.relpath(key, s3_prefix)
            local_path = os.path.join(local_folder, rel_path)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            print(f"Downloading s3://{bucket}/{key} → {local_path}")
            s3.download_file(bucket, key, local_path)
    print("DONE: Folder downloaded successfully")

download_folder(BUCKET, S3_PREFIX, LOCAL_FOLDER)